# 利用Unstructured工具基于文档结构分块

Unstructured工具中有两种主要的分块策略----Basic和By Title。这两种策略都基于对文档语义结构的识别， 而非简单地根据空行或换行符来拆分文本

- Basic策略将文本元素顺序合并到同一个分块内，直至达到最大字符数（max_characters）或软限制（new_after_n_chars）。如果单个元素（如一段特别长的正文或一张很大的表格） 本身超过了最大字符数， 则会对该元素进行二次拆分。 表格元素会被视为独立的分块； 如果表格过大， 也会被拆分为多个TableChunk可以通过overlap与overlap_all等参数设置分块重叠
- By Title策略在保留Basic策略的基础行为的同时，会在检测到新的标题（Title元素）后立刻关闭当前分块，并开启一个新的分块。可以通过multipage_sections和combine_text_under_n_chars等参数进一步控制如何合并或拆分跨页片段、 短小片段等

In [2]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    "../data/黑悟空/设定.txt",
    chunking_strategy="basic",  # 指定分块策略为 Basic
    max_characters=1000,
    include_orig_elements=False,  #不保留原始元素信息， 只保留合并后的文本块
)

docs=loader.load()
print("分块后LangChain的Document 数量:",len(docs))
print("第1个分块的文本长度:",len(docs[0].page_content))

分块后LangChain的Document 数量: 1
第1个分块的文本长度: 213


In [3]:
loader=UnstructuredLoader(
    "../data/黑悟空/设定.txt",
    chunking_strategy="by_title" #指定分块策略为By Title，按标题进行分块
)

docs=loader.load()
print("分块后LangChain的Document 数量:",len(docs))
print("第1个分块的文本长度:",len(docs[0].page_content))

分块后LangChain的Document 数量: 1
第1个分块的文本长度: 213


## 利用Llamalndex的SemanticSplitterNodeParser进行语义分块

传统的文本分块通常是基于固定字符数（如每 500 个字切一刀），这往往会将一个完整的句子或逻辑段落从中间切断，导致 AI 在检索时丢失上下文。

SemanticSplitterNodeParser 的核心逻辑是：“只有当意思发生变化时，才进行切分。”

__工作原理:__

    - 嵌入 (Embedding)：它会将文本按句子拆分，并为每个句子生成向量。

    - 相似度计算：计算相邻句子之间的余弦相似度。

    - 断点识别：如果两个相邻句子的向量差异很大（低于设定的阈值），说明话题发生了转换，它就会在这里设置一个切分点。

```
pip install llama-index-embeddings-dashscope llama-index-llms-dashscope
```

In [5]:
import os
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.dashscope import DashScopeEmbedding

# 1. 配置 API Key
os.environ["DASHSCOPE_API_KEY"] = "sk-51b422ad7151406b8c3ddb1ce0a424ba"

# 2. 初始化千问 Embedding
embed_model = DashScopeEmbedding(model_name="text-embedding-v2")

# 3. 只加载特定的文件
# 可以传入一个列表，包含一个或多个文件的绝对路径或相对路径
specific_files = ["../data/黑悟空/设定.txt"]

# 使用 input_files 参数直接指定文件
reader = SimpleDirectoryReader(input_files=specific_files)
documents = reader.load_data()

# 4. 配置语义分块器
splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model
)

# 5. 生成节点
nodes = splitter.get_nodes_from_documents(documents)

print(f"成功从指定文件加载，生成了 {len(nodes)} 个语义分块。")

成功从指定文件加载，生成了 1 个语义分块。
